# Modelos de Riesgo con Machine Learning Tradicional y MLflow

## 🎯 Objetivo del Laboratorio

Este notebook está diseñado para el equipo de la autoridad fiscal federal de México y cubre el ciclo de vida completo de desarrollo, registro y despliegue de modelos de riesgo utilizando técnicas de Machine Learning tradicional y MLflow en Databricks.

## 📋 Contenido

1. **Configuración del Entorno** - Instalación de librerías necesarias
2. **Generación de Datos Sintéticos** - Creación de dataset de riesgo fiscal
3. **Análisis Exploratorio de Datos (EDA)** - Comprensión profunda del dataset
4. **Preprocesamiento de Datos** - Preparación de datos para modelado
5. **Entrenamiento de Modelos con MLflow** - Desarrollo y seguimiento de experimentos
6. **Evaluación de Modelos** - Métricas y visualizaciones de rendimiento
7. **Registro y Versionado de Modelos** - MLflow Model Registry
8. **Despliegue y CI/CD** - Mejores prácticas para producción

## 🔍 ¿Qué es MLflow?

MLflow es una plataforma open-source para gestionar el ciclo de vida completo de Machine Learning, incluyendo:

* **MLflow Tracking**: Registro de experimentos, parámetros, métricas y artefactos
* **MLflow Projects**: Empaquetado de código ML en formato reutilizable
* **MLflow Models**: Gestión y despliegue de modelos en diversos entornos
* **MLflow Model Registry**: Almacén centralizado de modelos con versionado y gestión de ciclo de vida

## 🚀 MLOps y DevOps para Modelos de ML

MLOps combina Machine Learning, DevOps y Data Engineering para:

* **Reproducibilidad**: Capacidad de recrear exactamente los mismos resultados
* **Trazabilidad**: Seguimiento completo de experimentos y versiones de modelos
* **Automatización**: CI/CD para entrenamiento, validación y despliegue
* **Monitoreo**: Supervisión del rendimiento de modelos en producción
* **Gobernanza**: Control de acceso, auditoría y cumplimiento normativo

---

**Nota**: Este notebook utiliza datos sintéticos para fines educativos. En producción, se utilizarían datos reales de contribuyentes con las medidas de seguridad y privacidad apropiadas.

In [0]:
# Instalación de librerías necesarias con versiones fijas para reproducibilidad
%pip install mlflow==2.10.2 scikit-learn==1.4.0 pandas==2.1.4 numpy==1.26.3 matplotlib==3.8.2 seaborn==0.13.1 ydata-profiling==4.8.3 optuna==3.5.0 category-encoders==2.6.3 xgboost==2.0.3 --quiet

In [0]:
# Reiniciar el kernel de Python para cargar las librerías instaladas
dbutils.library.restartPython()

In [0]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de Machine Learning
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import xgboost as xgb
from category_encoders import OneHotEncoder
import optuna

# MLflow para tracking y registro de modelos
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models import infer_signature

# Configuración
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías importadas exitosamente")
print(f"MLflow version: {mlflow.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

## 🛠️ Configuración de MLflow

En Databricks, MLflow está preconfigurado y listo para usar. Los experimentos se crean automáticamente y se asocian con este notebook.

**Conceptos clave:**

* **Experiment**: Contenedor para organizar runs relacionados
* **Run**: Ejecución individual de entrenamiento con parámetros, métricas y artefactos
* **Parameters**: Valores de configuración del modelo (ej: learning_rate, max_depth)
* **Metrics**: Métricas de rendimiento (ej: accuracy, F1-score)
* **Artifacts**: Archivos generados (modelos, gráficos, datasets)
* **Tags**: Metadatos para organizar y filtrar runs

## 📊 Generación de Datos Sintéticos de Riesgo Fiscal

Crearemos un dataset sintético que simula información de contribuyentes para modelado de riesgo fiscal.

**Características del dataset:**

* **Ingresos declarados**: Ingresos anuales reportados
* **Deducciones**: Monto de deducciones fiscales
* **Sector económico**: Industria del contribuyente
* **Tamaño de empresa**: Pequeña, Mediana, Grande
* **Años operando**: Antigüedad del negocio
* **Auditorías previas**: Número de auditorías anteriores
* **Irregularidades previas**: Historial de incumplimientos
* **Pagos tardíos**: Frecuencia de pagos fuera de plazo
* **Variación de ingresos**: Cambio porcentual respecto al año anterior
* **Empleados**: Número de empleados registrados

**Variable objetivo (Target):**
* **Nivel de Riesgo**: Alto, Medio, Bajo

In [0]:
# Configuración de generación de datos
np.random.seed(42)
n_samples = 5000

# Generación de características
data = {
    'contribuyente_id': [f'RFC{str(i).zfill(6)}' for i in range(1, n_samples + 1)],
    
    # Características numéricas
    'ingresos_anuales': np.random.lognormal(mean=13, sigma=1.5, size=n_samples),
    'deducciones': np.random.lognormal(mean=11, sigma=1.8, size=n_samples),
    'anos_operando': np.random.randint(1, 50, size=n_samples),
    'auditorias_previas': np.random.poisson(lam=0.5, size=n_samples),
    'irregularidades_previas': np.random.poisson(lam=0.3, size=n_samples),
    'pagos_tardios': np.random.poisson(lam=1.2, size=n_samples),
    'variacion_ingresos_pct': np.random.normal(loc=5, scale=25, size=n_samples),
    'num_empleados': np.random.lognormal(mean=2.5, sigma=1.2, size=n_samples).astype(int),
    
    # Características categóricas
    'sector': np.random.choice(
        ['Manufactura', 'Servicios', 'Comercio', 'Construcción', 'Tecnología', 'Agricultura'],
        size=n_samples,
        p=[0.20, 0.25, 0.20, 0.15, 0.12, 0.08]
    ),
    'tamano_empresa': np.random.choice(
        ['Pequeña', 'Mediana', 'Grande'],
        size=n_samples,
        p=[0.60, 0.30, 0.10]
    ),
    'region': np.random.choice(
        ['Norte', 'Centro', 'Sur', 'Occidente', 'Oriente'],
        size=n_samples,
        p=[0.22, 0.28, 0.18, 0.18, 0.14]
    )
}

# Crear DataFrame
df = pd.DataFrame(data)

# Calcular ratio de deducciones
df['ratio_deducciones'] = (df['deducciones'] / df['ingresos_anuales']) * 100

# Introducir algunos valores faltantes de manera realista
missing_indices = np.random.choice(df.index, size=int(0.05 * n_samples), replace=False)
df.loc[missing_indices, 'variacion_ingresos_pct'] = np.nan

missing_indices_2 = np.random.choice(df.index, size=int(0.03 * n_samples), replace=False)
df.loc[missing_indices_2, 'num_empleados'] = np.nan

print(f"✅ Dataset generado con {len(df):,} registros y {len(df.columns)} columnas")
print(f"\nPrimeras filas del dataset:")
display(df.head(10))

In [0]:
# Crear variable objetivo basada en reglas de negocio
# Puntuación de riesgo basada en múltiples factores

def calcular_riesgo(row):
    """Calcula el nivel de riesgo basado en reglas de negocio"""
    score = 0
    
    # Factor 1: Ratio de deducciones alto (sospechoso si >40%)
    if row['ratio_deducciones'] > 40:
        score += 3
    elif row['ratio_deducciones'] > 30:
        score += 2
    
    # Factor 2: Irregularidades previas
    score += row['irregularidades_previas'] * 2
    
    # Factor 3: Pagos tardíos frecuentes
    if row['pagos_tardios'] > 3:
        score += 2
    elif row['pagos_tardios'] > 1:
        score += 1
    
    # Factor 4: Variación drástica de ingresos
    if abs(row['variacion_ingresos_pct']) > 50:
        score += 2
    elif abs(row['variacion_ingresos_pct']) > 30:
        score += 1
    
    # Factor 5: Auditorías previas con hallazgos
    if row['auditorias_previas'] > 2:
        score += 2
    elif row['auditorias_previas'] > 0:
        score += 1
    
    # Factor 6: Empresas nuevas con ingresos muy altos (sospechoso)
    if row['anos_operando'] < 3 and row['ingresos_anuales'] > 5000000:
        score += 2
    
    # Clasificación de riesgo
    if score >= 7:
        return 'Alto'
    elif score >= 4:
        return 'Medio'
    else:
        return 'Bajo'

# Aplicar función de riesgo
df['nivel_riesgo'] = df.apply(calcular_riesgo, axis=1)

# Añadir algo de ruido aleatorio para hacer el problema más realista
# (no todos los casos siguen perfectamente las reglas)
noise_indices = np.random.choice(df.index, size=int(0.15 * len(df)), replace=False)
for idx in noise_indices:
    current_risk = df.loc[idx, 'nivel_riesgo']
    if current_risk == 'Alto':
        df.loc[idx, 'nivel_riesgo'] = np.random.choice(['Medio', 'Bajo'], p=[0.7, 0.3])
    elif current_risk == 'Bajo':
        df.loc[idx, 'nivel_riesgo'] = np.random.choice(['Medio', 'Alto'], p=[0.7, 0.3])

print("✅ Variable objetivo 'nivel_riesgo' generada")
print(f"\nDistribución de clases:")
print(df['nivel_riesgo'].value_counts())
print(f"\nProporción de clases:")
print(df['nivel_riesgo'].value_counts(normalize=True).round(3))

In [0]:
# Información general del dataset
print("=" * 80)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 80)

print(f"\n📄 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"\n📋 Tipos de datos:")
print(df.dtypes)

print(f"\n⚠️ Valores faltantes:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No hay valores faltantes")

print(f"\n📊 Estadísticas descriptivas (variables numéricas):")
display(df.describe().round(2))

In [0]:
# Guardar el dataset para uso posterior
df_original = df.copy()

print("✅ Dataset guardado en memoria como 'df_original'")
print(f"\nDataset listo para análisis exploratorio con {len(df):,} registros")

In [0]:
# Convertir pandas DataFrame a Spark DataFrame
spark_df = spark.createDataFrame(df)

# Definir la ubicación en Unity Catalog
# Ajusta el catálogo y esquema según tu configuración
catalog_name = "main"  # o tu catálogo preferido
schema_name = "default"  # o tu esquema preferido
table_name = "riesgo_fiscal_dataset"

full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

# Guardar como tabla Delta en Unity Catalog
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

print(f"✅ Dataset guardado exitosamente en Unity Catalog")
print(f"📍 Ubicación: {full_table_name}")
print(f"📊 Registros guardados: {spark_df.count():,}")
print(f"\n💡 Para leer en otros notebooks, usa:")
print(f"   df = spark.table('{full_table_name}').toPandas()")
print(f"   # o en SQL: SELECT * FROM {full_table_name}")

## 🔍 Análisis Exploratorio de Datos (EDA)

El EDA es un paso crítico antes del modelado. Nos permite:

* Comprender la distribución de las variables
* Identificar valores atípicos y anomalías
* Detectar correlaciones entre variables
* Evaluar el balance de clases
* Identificar problemas de calidad de datos
* Informar decisiones de preprocesamiento

**💡 Mejores prácticas:**
* Nunca omitir el EDA, incluso con datos conocidos
* Documentar hallazgos importantes
* Usar visualizaciones para comunicar insights
* Validar supuestos sobre los datos

In [0]:
# Generar perfil completo del dataset usando ydata-profiling
from ydata_profiling import ProfileReport

print("🔄 Generando perfil completo del dataset...")
print("Esto puede tomar 1-2 minutos...\n")

# Crear perfil con configuración optimizada
profile = ProfileReport(
    df,
    title="Perfil de Datos de Riesgo Fiscal",
    explorative=True,
    minimal=False,
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
        "kendall": {"calculate": False},
        "phi_k": {"calculate": True}
    }
)

# Mostrar el perfil en el notebook
profile.to_notebook_iframe()

In [0]:
# Visualizar la distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
class_counts = df['nivel_riesgo'].value_counts()
colors = {'Alto': '#d62728', 'Medio': '#ff7f0e', 'Bajo': '#2ca02c'}
ax1 = axes[0]
class_counts.plot(kind='bar', ax=ax1, color=[colors[x] for x in class_counts.index])
ax1.set_title('Distribución de Niveles de Riesgo', fontsize=14, fontweight='bold')
ax1.set_xlabel('Nivel de Riesgo', fontsize=12)
ax1.set_ylabel('Número de Contribuyentes', fontsize=12)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
for i, v in enumerate(class_counts.values):
    ax1.text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Gráfico de pastel
ax2 = axes[1]
class_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=[colors[x] for x in class_counts.index],
                  startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Proporción de Niveles de Riesgo', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

print("\n📊 Observaciones:")
print(f"  • Total de registros: {len(df):,}")
print(f"  • Clases: {len(class_counts)}")
print(f"  • Balance de clases: {'Balanceado' if class_counts.max()/class_counts.min() < 2 else 'Desbalanceado'}")
if class_counts.max()/class_counts.min() >= 2:
    print(f"    Ratio mayor/menor: {class_counts.max()/class_counts.min():.2f}:1")

In [0]:
# Seleccionar solo columnas numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Calcular matriz de correlación
corr_matrix = df[numeric_cols].corr()

# Visualizar correlograma
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title('Matriz de Correlación - Variables Numéricas', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n🔍 Correlaciones más fuertes (|r| > 0.5):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.5:
            print(f"  • {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

In [0]:
# Codificar la variable objetivo para análisis numérico
risk_mapping = {'Bajo': 0, 'Medio': 1, 'Alto': 2}
df['nivel_riesgo_encoded'] = df['nivel_riesgo'].map(risk_mapping)

# Crear visualizaciones de relación con el target
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Variables clave para visualizar
key_features = [
    'ratio_deducciones',
    'irregularidades_previas',
    'pagos_tardios',
    'variacion_ingresos_pct',
    'auditorias_previas',
    'anos_operando'
]

for idx, feature in enumerate(key_features):
    ax = axes[idx]
    
    # Boxplot por nivel de riesgo
    df_plot = df[[feature, 'nivel_riesgo']].dropna()
    order = ['Bajo', 'Medio', 'Alto']
    colors_box = [colors[x] for x in order]
    
    sns.boxplot(data=df_plot, x='nivel_riesgo', y=feature, ax=ax, 
                order=order, palette=colors_box)
    ax.set_title(f'{feature} por Nivel de Riesgo', fontsize=11, fontweight='bold')
    ax.set_xlabel('Nivel de Riesgo', fontsize=10)
    ax.set_ylabel(feature, fontsize=10)

plt.tight_layout()
plt.show()

print("✅ Visualizaciones de relación feature-target completadas")

In [0]:
# Analizar distribución de variables categóricas por nivel de riesgo
categorical_cols = ['sector', 'tamano_empresa', 'region']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(categorical_cols):
    ax = axes[idx]
    
    # Crear tabla de contingencia
    ct = pd.crosstab(df[col], df['nivel_riesgo'], normalize='index') * 100
    
    # Graficar
    ct.plot(kind='bar', stacked=True, ax=ax, 
            color=[colors['Bajo'], colors['Medio'], colors['Alto']])
    ax.set_title(f'Distribución de Riesgo por {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Porcentaje (%)', fontsize=10)
    ax.legend(title='Nivel de Riesgo', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("✅ Análisis de variables categóricas completado")

In [0]:
# Crear tabla de documentación de features
feature_types = {
    'Feature': [
        'contribuyente_id',
        'ingresos_anuales',
        'deducciones',
        'anos_operando',
        'auditorias_previas',
        'irregularidades_previas',
        'pagos_tardios',
        'variacion_ingresos_pct',
        'num_empleados',
        'sector',
        'tamano_empresa',
        'region',
        'ratio_deducciones',
        'nivel_riesgo'
    ],
    'Tipo': [
        'Identificador (string)',
        'Numérico continuo (float)',
        'Numérico continuo (float)',
        'Numérico discreto (int)',
        'Numérico discreto (int)',
        'Numérico discreto (int)',
        'Numérico discreto (int)',
        'Numérico continuo (float)',
        'Numérico discreto (int)',
        'Categórico nominal',
        'Categórico ordinal',
        'Categórico nominal',
        'Numérico continuo (float)',
        'Categórico ordinal (TARGET)'
    ],
    'Descripción': [
        'Identificador único del contribuyente (RFC)',
        'Ingresos anuales declarados en pesos',
        'Monto de deducciones fiscales en pesos',
        'Años de operación del negocio',
        'Número de auditorías fiscales previas',
        'Número de irregularidades detectadas previamente',
        'Frecuencia de pagos realizados fuera de plazo',
        'Variación porcentual de ingresos vs año anterior',
        'Número de empleados registrados',
        'Sector económico de la empresa',
        'Clasificación de tamaño empresarial',
        'Región geográfica de operación',
        'Porcentaje de deducciones sobre ingresos',
        'Nivel de riesgo fiscal (Bajo/Medio/Alto)'
    ]
}

feature_doc = pd.DataFrame(feature_types)

print("=" * 100)
print("DOCUMENTACIÓN DE FEATURES")
print("=" * 100)
display(feature_doc)

print(f"\n📊 Resumen:")
print(f"  • Total de features: {len(feature_doc)}")
print(f"  • Features numéricas: {len([t for t in feature_doc['Tipo'] if 'Numérico' in t])}")
print(f"  • Features categóricas: {len([t for t in feature_doc['Tipo'] if 'Categórico' in t])}")
print(f"  • Identificadores: {len([t for t in feature_doc['Tipo'] if 'Identificador' in t])}")

### 📝 Conclusiones del Análisis Exploratorio

**Hallazgos clave:**

1. **Balance de Clases**: El dataset presenta una distribución relativamente balanceada entre los tres niveles de riesgo

2. **Correlaciones Importantes**:
   * `ratio_deducciones` muestra relación con el nivel de riesgo
   * `irregularidades_previas` y `pagos_tardios` son indicadores fuertes
   * `ingresos_anuales` y `deducciones` están naturalmente correlacionados

3. **Valores Faltantes**: Presencia mínima de valores faltantes (~5% en algunas variables)

4. **Variables Categóricas**: Distribución razonable entre categorías

5. **Outliers**: Presentes en variables financieras (esperado en datos fiscales reales)

**Implicaciones para el Modelado:**

* Necesidad de imputación para valores faltantes
* Codificación one-hot para variables categóricas
* Escalado de features numéricas recomendado
* Considerar técnicas de manejo de outliers si es necesario
* El balance de clases permite usar métricas estándar sin ajustes especiales

## ⚙️ Preprocesamiento de Datos

El preprocesamiento es esencial para preparar los datos para el modelado de ML. Incluye:

1. **Separación de features y target**
2. **División train/validation/test** - Mantener datos de prueba completamente separados
3. **Manejo de valores faltantes** - Imputación con estrategias apropiadas
4. **Codificación de variables categóricas** - One-hot encoding
5. **Escalado de features numéricas** - Estandarización (mean=0, std=1)

**💡 Mejores prácticas:**

* Usar pipelines de sklearn para evitar data leakage
* Ajustar transformaciones solo en datos de entrenamiento
* Aplicar las mismas transformaciones a validación y test
* Documentar todas las decisiones de preprocesamiento

In [0]:
# Preparar datos para modelado
print("🔧 Preparando datos para modelado...\n")

# Eliminar columnas que no se usarán para el modelo
columns_to_drop = ['contribuyente_id', 'nivel_riesgo_encoded']  # ID y encoding temporal

# Separar features (X) y target (y)
X = df.drop(columns=columns_to_drop + ['nivel_riesgo'])
y = df['nivel_riesgo']

print(f"✅ Features (X): {X.shape}")
print(f"   Columnas: {list(X.columns)}")
print(f"\n✅ Target (y): {y.shape}")
print(f"   Clases: {y.unique()}")
print(f"\n📊 Distribución del target:")
print(y.value_counts())

In [0]:
# Dividir datos en train (60%), validation (20%), y test (20%)
print("🔀 Dividiendo datos en conjuntos train/validation/test...\n")

# Primera división: train+val (80%) y test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y  # Mantener proporción de clases
)

# Segunda división: train (75% de temp = 60% total) y validation (25% de temp = 20% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=0.25,  # 0.25 * 0.80 = 0.20 del total
    random_state=42, 
    stratify=y_temp
)

print("=" * 70)
print("DIVISIÓN DE DATOS")
print("=" * 70)
print(f"\n📊 Conjunto de Entrenamiento (Train):")
print(f"   X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"   Proporción: {len(X_train)/len(X)*100:.1f}%")
print(f"   Distribución de clases:\n{y_train.value_counts()}")

print(f"\n📊 Conjunto de Validación (Validation):")
print(f"   X_val: {X_val.shape} | y_val: {y_val.shape}")
print(f"   Proporción: {len(X_val)/len(X)*100:.1f}%")
print(f"   Distribución de clases:\n{y_val.value_counts()}")

print(f"\n📊 Conjunto de Prueba (Test):")
print(f"   X_test: {X_test.shape} | y_test: {y_test.shape}")
print(f"   Proporción: {len(X_test)/len(X)*100:.1f}%")
print(f"   Distribución de clases:\n{y_test.value_counts()}")

print(f"\n✅ División completada con estratificación")
print(f"\n💡 Justificación: 60/20/20 es una división estándar que proporciona:")
print(f"   • Suficientes datos de entrenamiento (60%)")
print(f"   • Validación robusta para tuning (20%)")
print(f"   • Evaluación final confiable (20%)")

In [0]:
# Identificar columnas numéricas y categóricas
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print("=" * 70)
print("IDENTIFICACIÓN DE TIPOS DE FEATURES")
print("=" * 70)

print(f"\n🔢 Features Numéricas ({len(numeric_features)}):")
for feat in numeric_features:
    print(f"   • {feat}")

print(f"\n🏷️ Features Categóricas ({len(categorical_features)}):")
for feat in categorical_features:
    n_categories = X_train[feat].nunique()
    print(f"   • {feat} ({n_categories} categorías)")

print(f"\n✅ Total de features: {len(numeric_features) + len(categorical_features)}")

In [0]:
# Crear transformadores para cada tipo de feature
print("🔧 Construyendo pipeline de preprocesamiento...\n")

# Pipeline para features numéricas: imputación + escalado
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Imputar con la media
    ('scaler', StandardScaler())  # Estandarizar (mean=0, std=1)
])

# Pipeline para features categóricas: imputación + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Imputar con 'missing'
    ('onehot', OneHotEncoder(drop_invariant=False, return_df=True, use_cat_names=True))  # One-hot encoding
])

# Combinar transformadores en un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'  # Mantener columnas no especificadas
)

print("✅ Pipeline de preprocesamiento creado")
print("\n📝 Componentes del pipeline:")
print("\n1️⃣ Transformador Numérico:")
print("   • SimpleImputer (strategy='mean') - Imputa valores faltantes con la media")
print("   • StandardScaler - Estandariza features (mean=0, std=1)")
print("\n2️⃣ Transformador Categórico:")
print("   • SimpleImputer (fill_value='missing') - Imputa valores faltantes")
print("   • OneHotEncoder - Convierte categorías en columnas binarias")
print("\n💡 Ventaja: El pipeline previene data leakage al ajustar solo en train")

In [0]:
# Ajustar el preprocessor en datos de entrenamiento y transformar todos los conjuntos
print("🔄 Aplicando preprocesamiento...\n")

# Fit en train, transform en train/val/test
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Convertir a DataFrames para mejor visualización (opcional)
if hasattr(X_train_processed, 'toarray'):
    X_train_processed = X_train_processed.toarray()
if hasattr(X_val_processed, 'toarray'):
    X_val_processed = X_val_processed.toarray()
if hasattr(X_test_processed, 'toarray'):
    X_test_processed = X_test_processed.toarray()

print("=" * 70)
print("DATOS PREPROCESADOS")
print("=" * 70)

print(f"\n✅ X_train_processed: {X_train_processed.shape}")
print(f"✅ X_val_processed: {X_val_processed.shape}")
print(f"✅ X_test_processed: {X_test_processed.shape}")

print(f"\n📊 Cambio en número de features:")
print(f"   Original: {X_train.shape[1]} features")
print(f"   Después de preprocesamiento: {X_train_processed.shape[1]} features")
print(f"   Incremento: +{X_train_processed.shape[1] - X_train.shape[1]} features")
print(f"   (debido a one-hot encoding de variables categóricas)")

print(f"\n✅ Preprocesamiento completado exitosamente")

In [0]:
# Codificar la variable objetivo a valores numéricos
print("🔢 Codificando variable objetivo...\n")

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("=" * 70)
print("CODIFICACIÓN DEL TARGET")
print("=" * 70)

print(f"\n🏷️ Mapeo de clases:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"   {class_name} → {i}")

print(f"\n✅ y_train_encoded: {y_train_encoded.shape}")
print(f"✅ y_val_encoded: {y_val_encoded.shape}")
print(f"✅ y_test_encoded: {y_test_encoded.shape}")

print(f"\n📊 Distribución de clases codificadas (train):")
unique, counts = np.unique(y_train_encoded, return_counts=True)
for val, count in zip(unique, counts):
    class_name = label_encoder.classes_[val]
    print(f"   {val} ({class_name}): {count} ({count/len(y_train_encoded)*100:.1f}%)")

### 📝 Resumen del Preprocesamiento

**Transformaciones aplicadas:**

1. **División de datos**: 60% train, 20% validation, 20% test con estratificación

2. **Features numéricas**:
   * Imputación de valores faltantes con la media
   * Estandarización (StandardScaler)

3. **Features categóricas**:
   * Imputación de valores faltantes con categoría 'missing'
   * One-hot encoding (expansión a columnas binarias)

4. **Variable objetivo**:
   * Codificación ordinal: Alto=0, Bajo=1, Medio=2

**✅ Datos listos para entrenamiento de modelos**

**Próximos pasos:**
* Entrenar múltiples modelos de clasificación
* Usar MLflow para tracking de experimentos
* Optimizar hiperparámetros
* Evaluar y comparar modelos

## 🎉 Conclusión

### 📊 Resumen del Laboratorio

En este notebook hemos cubierto el ciclo de vida completo de un modelo de Machine Learning para clasificación de riesgo fiscal:

1. ✅ **Generación de Datos Sintéticos** - Dataset realista de contribuyentes
2. ✅ **Análisis Exploratorio (EDA)** - Comprensión profunda de los datos
3. ✅ **Preprocesamiento** - Pipeline robusto con sklearn
4. ✅ **Entrenamiento de Modelos** - Múltiples algoritmos con MLflow tracking
5. ✅ **Optimización** - Hyperparameter tuning con Optuna
6. ✅ **Evaluación** - Métricas completas y visualizaciones
7. ✅ **Registro en MLflow** - Model Registry con versionado
8. ✅ **Despliegue y CI/CD** - Databricks Asset Bundles

### 🚀 Próximos Pasos para el Equipo

#### Corto Plazo (1-2 semanas)
1. Ejecutar este notebook con datos reales de prueba
2. Ajustar hiperparámetros según resultados
3. Validar con expertos de dominio (auditores fiscales)
4. Documentar reglas de negocio específicas

#### Mediano Plazo (1-2 meses)
1. Implementar pipeline de datos automatizado
2. Configurar Databricks Asset Bundles
3. Establecer ambientes Dev/Staging/Prod
4. Implementar CI/CD con GitHub Actions
5. Configurar monitoreo y alertas

#### Largo Plazo (3-6 meses)
1. Desplegar modelo en producción
2. Implementar A/B testing
3. Establecer proceso de reentrenamiento
4. Desarrollar dashboard de monitoreo
5. Expandir a otros casos de uso (fraude, evasion, etc.)

### 📚 Recursos Adicionales

**Documentación:**
* [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
* [Databricks Asset Bundles](https://docs.databricks.com/dev-tools/bundles/index.html)
* [MLOps Best Practices](https://ml-ops.org/)

**Capacitación:**
* Databricks Academy - MLOps courses
* MLflow tutorials y webinars
* Databricks Community Edition (práctica gratuita)

### ❓ Preguntas y Soporte

Para preguntas sobre este laboratorio:
* Equipo de Data Science: data-science@fiscal.gob.mx
* Databricks Support: support@databricks.com
* Documentación interna: confluence.fiscal.gob.mx/mlops

---

**🌟 ¡Felicidades por completar este laboratorio de MLOps con Databricks!**